# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back upward (cat) and letting it sag downward (cow). Aim for 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg simultaneously while engaging your core. Hold each extension for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor briefly before lowering. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back and pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30 seconds, then switch

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (typically 7-9 hours for adults) enhances immune function, helps manage stress, and maintains hormonal balance. Poor or insufficient sleep can lead to various health issues, including weakened immunity, increased risk of chronic conditions, mental health problems, and impaired cognitive performance. Therefore, maintaining good sleep hygiene and a consistent sleep routine is essential for promoting overall health and well-being.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated to help alleviate dehydration-related headaches.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gentle massage of the temples and neck.\n- Using essential oils such as peppermint or lavender.\n- Engaging in deep breathing exercises, progressive muscle relaxation, or grounding techniques for immediate stress relief.\n- Taking short walks, especially in nature, and listening to calming music.\n- Maintaining a regular sleep schedule and practicing relaxation techniques like meditation.\n- Light stretching or yoga, warm baths, reading, journaling, and gratitude practices as part of evening routines.\n- Limiting stimulant intake and managing environmental triggers.\n\nThese approaches can help manage stress and headaches naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate arching your back up (cat) and letting it sag down (cow). Repeat 10-15 times.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds, then repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Maintaining a consistent and quality sleep schedule—a key aspect of good sleep hygiene—helps promote restorative rest. Creating an optimal sleep environment by keeping the room cool, dark, and quiet, and establishing relaxing bedtime routines can enhance sleep quality. Adequate sleep supports various bodily functions, including immune function, mental health, and physical recovery. Poor or insufficient sleep can contribute to health issues such as insomnia, weakened immunity, and mental health challenges. Therefore, prioritizing good sleep habits is essential for overall wellness.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root can promote relaxation and reduce headache symptoms. Ensuring adequate hydration, managing stress through relaxation methods, and maintaining good sleep habits can also help alleviate headaches and reduce stress levels.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

*Among the three demo queries, “What are some natural remedies for stress and headaches?” is the one where BM25 can be better than embeddings. That question uses clear, specific terms (“natural remedies”, “stress”, “headaches”) that likely appear in the guide, so BM25’s keyword matching surfaces the most relevant, term-heavy chunks. The BM25 answer’s mention of “herbal teas like chamomile or valerian root” suggests it pulled chunks that contain those exact phrases, which BM25 rewards. The other two queries are less favorable for BM25: the lower-back-pain question gets a more complete exercise list from the naive (embedding) chain, and the sleep question is broad and conceptual, where embeddings usually do better.*

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Repeat 10-15 times.\n- Bird Dog: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—supports physical health, mental well-being, and cognitive function. It involves cycles of REM and non-REM sleep, with deep sleep (Stage 3) being particularly important for bodily repair and regeneration. Ideally, creating a comfortable sleep environment—appropriate temperature, darkness, quiet, and quality bedding—can enhance sleep quality and, consequently, overall health.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, engaging in progressive muscle relaxation, using grounding techniques, taking short walks in nature, listening to calming music, drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the information provided, exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each side for about 5 seconds. Perform 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, and repeat 8-12 times.\n\nAlways remember to perform these e

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours per night for adults) and good sleep quality are essential for maintaining a healthy immune system, managing stress, and ensuring optimal mental and physical functioning. Poor sleep or sleep disorders like insomnia can negatively impact health, leading to issues such as weakened immune response, increased stress, and cognitive impairments. Therefore, practicing good sleep hygiene and establishing consistent routines are crucial for overall health and wellness.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks in nature, and listening to calming music. \n\nFor headaches, natural remedies include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule. \n\nThese approaches can help manage stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

*In a RAG system, a multi-query retriever creates several different versions of the same user question. Each version uses slightly different words or phrasing. This helps because the right documents might not use the exact same words as the user, they may describe the same concept in different ways, and a single query might miss some relevant information. By running multiple variations of the query, the system increases the chance of matching more relevant documents. This improves recall, since the retriever is less likely to miss useful content that is phrased differently.*

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: On hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg, hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest, hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep has a significant impact on overall health. It is essential for the body's physical well-being, mental health, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—supports these vital processes and helps maintain optimal health. Poor sleep or sleep problems like insomnia can lead to various health issues, including fatigue, weakened immune function, and impaired cognitive performance. Practicing good sleep hygiene and creating a conducive sleep environment can improve sleep quality and, consequently, overall health."

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in relaxation techniques like progressive muscle relaxation, doing light stretching or yoga, taking warm baths, practicing mindfulness or meditation, and using essential oils such as peppermint or lavender. Additionally, staying well-hydrated by drinking water, maintaining a consistent sleep schedule, and managing stress through hobbies and social connections can help reduce both stress and headache symptoms.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'The exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, then repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It is essential for physical recovery, mental well-being, and cognitive functions such as memory and learning. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—is linked to benefits such as stronger immune function, improved mood, better stress management, and a reduced risk of chronic conditions. Conversely, poor sleep quality or insufficient sleep can lead to health problems like increased stress, weakened immune response, cognitive impairments, and a higher risk of chronic illnesses. Practicing good sleep hygiene, creating an optimal sleep environment, and managing stress are important strategies to support healthy sleep and, consequently, overall health.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises, such as inhaling for 4 counts, holding for 4, and exhaling for 4.\n- Progressive muscle relaxation, which involves tensing and releasing muscle groups from toes to head.\n- Grounding techniques, like naming sensory experiences (5 things you see, 4 you hear, etc.).\n- Taking short walks, preferably in nature.\n- Listening to calming music.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gentle massage of temples and neck.\n- Using essential oils such as peppermint or lavender.\n- Maintaining a regular sleep schedule.\n- Staying well-hydrated by drinking water.\n\nThese methods can help alleviate headaches and reduce stress naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of sentences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Begin on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises focus on gentle stretching and strengthening to alleviate discomfort and prevent future episodes of lower back pain.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in multiple ways. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—supports physical health, mental well-being, and cognitive function. Proper sleep hygiene practices, such as maintaining a consistent sleep schedule and creating a relaxing sleep environment, promote better sleep quality. Conversely, poor sleep or sleep disorders like insomnia can lead to issues such as fatigue, headaches, irritability, and weakened immune function, which can negatively affect overall health.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4, and hold for 4 to promote relaxation.\n- Progressive muscle relaxation: Tense and then release muscle groups progressively from toes to head.\n- Grounding techniques: Identify five things you see, four you hear, three you feel, two you smell, and one you taste to reduce stress.\n- Rest in a dark, quiet room and avoid screen exposure before bed to help manage headaches related to stress.\n- Gentle massage of temples and neck areas to relieve tension.\n- Use of essential oils like peppermint and lavender, which can help alleviate headache symptoms.\n- Staying hydrated by drinking plenty of water.\n- Applying warm or cold compresses to the head or neck.\n- Engaging in regular physical activity and maintaining good sleep hygiene.\n\nThese approaches are often recommended as natural ways to reduce stress and headaches. If symptoms persist, consulting a healt

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

*When sentences are short and very repetitive, semantic chunking has a hard time knowing where to split the text. Since many sentences sound almost the same, their similarity is high, the system cannot clearly see where one idea ends and another begins. Because of that, it may create a few big chunks that repeat similar information, or many small chunks that are almost identical. In both cases, the chunks are not very helpful when trying to find specific information. To improve this, I would try adjustusting the sentence distance threshold. If the chunks become too large, I would lower the percentile threshold so the system splits the text more easily. If the chunks are too small, I would increase the percentile threshold so more similar sentences stay grouped together. Also, I'd try enforcing minimum and maximum chunk size to prevent creating chunks that are either too short or too long.*

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key (optional, for tracing):")

In [54]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

In [57]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"golden_dataset"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [61]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextRecall, ContextEntityRecall, NoiseSensitivity, ContextPrecision
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

custom_run_config = RunConfig(timeout=360)

retrievers = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "contextual_compression": contextual_compression_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain,
    "ensemble": ensemble_retrieval_chain
}
results = {}
for name, graph in retrievers.items():
    for test_row in dataset:
        response = graph.invoke({"question" : test_row.eval_sample.user_input})
        msg = response["response"]
        test_row.eval_sample.response = msg.content if hasattr(msg, "content") else str(msg)
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

    evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())
    result = evaluate(
    dataset=evaluation_dataset,
    metrics=[ContextRecall(), ContextEntityRecall(), NoiseSensitivity(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
    )
    results[name] = result

results
    

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[14]: TimeoutError()
Exception raised in Job[26]: TimeoutError()


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a26d100> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-9144' coro=<_async_in_context.<locals>.run_in_context() running at /Users/karlaudiljak/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-9161' coro=<Kernel.shell_main() running at /Users/karlaudiljak/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/karlaudiljak/AIE9/11_Advanced_Retrieval/.ve

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[30]: TimeoutError()
Exception raised in Job[34]: TimeoutError()


{'naive': {'context_recall': 0.8571, 'context_entity_recall': 0.2277, 'noise_sensitivity_relevant': 0.1886, 'context_precision': 0.7747},
 'bm25': {'context_recall': 0.1270, 'context_entity_recall': 0.1270, 'noise_sensitivity_relevant': 0.0796, 'context_precision': 0.2222},
 'contextual_compression': {'context_recall': 0.7817, 'context_entity_recall': 0.2854, 'noise_sensitivity_relevant': 0.0711, 'context_precision': 0.6481},
 'multi_query': {'context_recall': 0.9841, 'context_entity_recall': 0.2787, 'noise_sensitivity_relevant': 0.0944, 'context_precision': 0.5963},
 'parent_document': {'context_recall': 0.9444, 'context_entity_recall': 0.2117, 'noise_sensitivity_relevant': 0.1938, 'context_precision': 1.0000},
 'ensemble': {'context_recall': 1.0000, 'context_entity_recall': 0.2833, 'noise_sensitivity_relevant': 0.0625, 'context_precision': 0.4275}}

In [63]:
from langsmith import evaluate

evaluate(
    naive_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "naive_retrieval_chain"},
)

View the evaluation results for experiment: 'tart-rate-14' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=0da22790-8fc0-435a-b869-2fb352cded30




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,"content='Based on Chapter 14, the main recomme...",[page_content='Sample wellness morning routine...,None,"Chapter 14 recomends that to support wellness,...",10.434798,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9074-c9b0-7cc2-8bb9-499846f9f782
1,what chapter 16 say for headaches?,"content='Chapter 16 of the guide is titled ""Ma...",[page_content='Chapter 16: Managing Headaches ...,None,Chapter 16 say headaches are common and you ca...,2.764740,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9074-f275-7623-bb20-2c4c2e9e4869
2,What practical strategies does PART 6 of the g...,content='PART 6 of the guide offers several pr...,[page_content='Chapter 16: Managing Headaches ...,None,PART 6 of the guide addresses common health co...,3.116767,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9074-fd43-7621-9487-eca4384c9faa
3,"Whaat aree the benfits of magnesum for sleep, ...","content=""Magnesium can benefit sleep by helpin...",[page_content='Types of insomnia:\n- Acute ins...,None,Magnesium supplements are listed as a natural ...,2.811605,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9075-0971-7061-9dd8-40965470ba53
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Types of insomnia:\n- Acute ins...,None,Chamomile can be used as an herbal tea as a na...,11.306364,a477005c-3955-4b00-86f4-d49345c480c3,019c9075-146d-7a32-8248-9a3819c84b29
5,How can meditation be incorporated into a busy...,"content=""Incorporating meditation into a busy ...",[page_content='Long-term stress management:\n-...,None,Meditation can be incorporated into a busy pro...,4.789375,ba56d476-e29d-461a-993b-defe23267a88,019c9075-4099-7af2-869b-ede094f3acb8
6,How do you do partial crunches for help with l...,"content=""Partial crunches can help strengthen ...",[page_content='- Partial Crunches: Lie on your...,None,Partial crunches is when you lie on your back ...,2.852793,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9075-534f-7de2-9803-c553e09582a5
7,Wht are Parshul Crunches and how do yu do them...,"content=""Parshul Crunches, as described in the...",[page_content='- Partial Crunches: Lie on your...,None,Partial Crunches are an exercise for lower bac...,3.930872,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9075-5e75-7f80-bdb0-3aef86fd6a8f
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,1.595354,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9075-6dd1-71a1-945f-49743828570e


In [64]:
evaluate(
    bm25_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "bm25_retrieval_chain"},
)


View the evaluation results for experiment: 'scholarly-color-76' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=22139a02-9fba-4cd3-a401-085128f24fd2




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,"content=""Based on Chapter 14, the main recomme...",[page_content='Weekly Meal Planning Steps:\n1....,None,"Chapter 14 recomends that to support wellness,...",2.971048,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9075-de0b-7f61-844b-23b8d07b5126
1,what chapter 16 say for headaches?,"content=""I don't see any information in the pr...",[page_content='Strategies for better balance:\...,None,Chapter 16 say headaches are common and you ca...,0.565812,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9075-e9a7-7eb3-8510-fbb474527f58
2,What practical strategies does PART 6 of the g...,"content=""The provided sections of the guide do...",[page_content='Types of insomnia:\n- Acute ins...,None,PART 6 of the guide addresses common health co...,3.273578,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9075-ebde-7881-ac8a-751a10bfe949
3,"Whaat aree the benfits of magnesum for sleep, ...","content=""Magnesium can be beneficial for sleep...",[page_content='Adults typically need 7-9 hours...,None,Magnesium supplements are listed as a natural ...,2.302284,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9075-f8a8-7621-b787-ca1c79c28723
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Types of insomnia:\n- Acute ins...,None,Chamomile can be used as an herbal tea as a na...,1.151413,a477005c-3955-4b00-86f4-d49345c480c3,019c9076-01a7-7f93-8f1e-fbfdb935dfdf
5,How can meditation be incorporated into a busy...,content='Incorporating meditation into a busy ...,[page_content='The Personal Wellness Guide\nA ...,None,Meditation can be incorporated into a busy pro...,3.759544,ba56d476-e29d-461a-993b-defe23267a88,019c9076-0628-7682-bfd5-e1b72b070f1e
6,How do you do partial crunches for help with l...,"content=""To do partial crunches that can help ...",[page_content='The Personal Wellness Guide\nA ...,None,Partial crunches is when you lie on your back ...,4.209958,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9076-14d8-7f82-b1a6-41fc2e40c049
7,Wht are Parshul Crunches and how do yu do them...,content='Parshul Crunches are a variation of a...,[page_content='The Personal Wellness Guide\nA ...,None,Partial Crunches are an exercise for lower bac...,2.547613,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9076-254b-7d42-908a-415efc569bcd
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,1.536095,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9076-2f40-7ad3-a78c-53d482198d22


In [65]:
evaluate(
    contextual_compression_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "contextual_compression_retrieval_chain"},
)

View the evaluation results for experiment: 'vacant-weather-95' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=e966e187-dfbc-4dad-b9d2-226084a106d5




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,content='Based on Chapter 14 of the Wellness G...,[page_content='Tips for building new habits:\n...,None,"Chapter 14 recomends that to support wellness,...",2.987412,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9076-3a61-70c2-be62-20e6c6bb6fb3
1,what chapter 16 say for headaches?,content='Chapter 16: Managing Headaches Natura...,[page_content='Chapter 15: Evening Wind-Down R...,None,Chapter 16 say headaches are common and you ca...,2.160414,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9076-460e-7c80-885b-c85e829ad36e
2,What practical strategies does PART 6 of the g...,"content='According to PART 6 of the guide, pra...",[page_content='Chapter 15: Evening Wind-Down R...,None,PART 6 of the guide addresses common health co...,4.341505,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9076-4e80-7b70-8dfe-92ddfb5edb68
3,"Whaat aree the benfits of magnesum for sleep, ...","content=""Magnesium may help improve sleep qual...",[page_content='Types of insomnia:\n- Acute ins...,None,Magnesium supplements are listed as a natural ...,2.401677,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9076-5f76-77e3-a7b0-251b40b70e91
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Types of insomnia:\n- Acute ins...,None,Chamomile can be used as an herbal tea as a na...,1.228050,a477005c-3955-4b00-86f4-d49345c480c3,019c9076-68d9-7d41-875e-c948033f53dd
5,How can meditation be incorporated into a busy...,"content=""To incorporate meditation into a busy...",[page_content='Long-term stress management:\n-...,None,Meditation can be incorporated into a busy pro...,3.545932,ba56d476-e29d-461a-993b-defe23267a88,019c9076-6da6-7cb1-ba4a-35687b20c3d2
6,How do you do partial crunches for help with l...,"content=""Partial crunches can help with lower ...",[page_content='- Partial Crunches: Lie on your...,None,Partial crunches is when you lie on your back ...,2.124274,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9076-7b81-7342-8bf1-fa93b8cc9918
7,Wht are Parshul Crunches and how do yu do them...,"content=""Parshul crunches, likely referring to...",[page_content='- Partial Crunches: Lie on your...,None,Partial Crunches are an exercise for lower bac...,3.492092,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9076-83ce-7660-8ec3-cb59a53d6ede
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,2.815850,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9076-9174-74a2-b92c-89b770a4bf65


In [66]:
evaluate(
    multi_query_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "multi_query_retrieval_chain"},
)

View the evaluation results for experiment: 'spotless-copper-50' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=3429e33e-829e-4d63-88ad-705d31f80de6




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,"content=""Based on Chapter 14 of the guide, the...",[page_content='Sample wellness morning routine...,None,"Chapter 14 recomends that to support wellness,...",14.287605,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9076-a310-7771-aa41-f61d110f1bff
1,what chapter 16 say for headaches?,content='Chapter 16 of the Health Wellness Gui...,[page_content='Chapter 16: Managing Headaches ...,None,Chapter 16 say headaches are common and you ca...,6.749342,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9076-dae1-7352-bebf-9eb0063fd936
2,What practical strategies does PART 6 of the g...,"content=""PART 6 of the guide offers several pr...",[page_content='Natural headache remedies:\n- D...,None,PART 6 of the guide addresses common health co...,13.180786,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9076-f53f-7a10-bb64-586a8a42b6cf
3,"Whaat aree the benfits of magnesum for sleep, ...",content='Magnesium can be beneficial for sleep...,[page_content='Chapter 8: Improving Sleep Qual...,None,Magnesium supplements are listed as a natural ...,3.714068,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9077-28bd-7fb1-ae18-6207cfe38f23
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Chapter 8: Improving Sleep Qual...,None,Chamomile can be used as an herbal tea as a na...,3.379136,a477005c-3955-4b00-86f4-d49345c480c3,019c9077-3741-7d92-94fb-c46a69c10beb
5,How can meditation be incorporated into a busy...,"content=""To incorporate meditation into a busy...",[page_content='Long-term stress management:\n-...,None,Meditation can be incorporated into a busy pro...,4.915310,ba56d476-e29d-461a-993b-defe23267a88,019c9077-4475-7f60-8fbf-3089581eac83
6,How do you do partial crunches for help with l...,content='To do partial crunches to help with l...,[page_content='Recommended exercises for lower...,None,Partial crunches is when you lie on your back ...,3.435851,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9077-57a9-7fd1-9125-a71a8d5e866f
7,Wht are Parshul Crunches and how do yu do them...,"content=""Parshul Crunches are a variation of p...",[page_content='- Partial Crunches: Lie on your...,None,Partial Crunches are an exercise for lower bac...,3.083643,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9077-6516-7b43-9620-e9a94749ee96
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,3.611827,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9077-7123-7d60-8b13-ae4be357f14a


In [67]:
evaluate(
    parent_document_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "parent_document_retrieval_chain"},
)

View the evaluation results for experiment: 'monthly-game-24' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=9e878278-bb1e-43b2-b46c-d31fc152a80b




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,"content='Based on Chapter 14, the main recomme...",[page_content='Types of meditation:\n- Focused...,None,"Chapter 14 recomends that to support wellness,...",2.452892,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9077-87ac-7ea1-8835-1be7a933d591
1,what chapter 16 say for headaches?,content='Chapter 16 of the provided document d...,[page_content='Chapter 15: Evening Wind-Down R...,None,Chapter 16 say headaches are common and you ca...,1.406828,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9077-9142-71f1-a515-560429b8c490
2,What practical strategies does PART 6 of the g...,content='PART 6 of the guide offers practical ...,[page_content='Chapter 15: Evening Wind-Down R...,None,PART 6 of the guide addresses common health co...,5.043038,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9077-96c2-7212-b433-afd52052a056
3,"Whaat aree the benfits of magnesum for sleep, ...","content=""Magnesium can be beneficial for sleep...",[page_content='Adults typically need 7-9 hours...,None,Magnesium supplements are listed as a natural ...,2.305392,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9077-aa76-7802-bbdf-a2f8af52d692
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Adults typically need 7-9 hours...,None,Chamomile can be used as an herbal tea as a na...,1.687539,a477005c-3955-4b00-86f4-d49345c480c3,019c9077-b379-7d61-8366-760acb977d37
5,How can meditation be incorporated into a busy...,"content=""Incorporating meditation into a busy ...",[page_content='PART 4: STRESS MANAGEMENT AND M...,None,Meditation can be incorporated into a busy pro...,4.417507,ba56d476-e29d-461a-993b-defe23267a88,019c9077-ba11-7580-9426-8fccf8da8983
6,How do you do partial crunches for help with l...,"content=""Partial crunches can help with lower ...",[page_content='The Personal Wellness Guide\nA ...,None,Partial crunches is when you lie on your back ...,5.716656,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9077-cb54-7bf0-aa0d-8d4bfc5ebc9c
7,Wht are Parshul Crunches and how do yu do them...,content='Parshul Crunches are likely a typo or...,[page_content='The Personal Wellness Guide\nA ...,None,Partial Crunches are an exercise for lower bac...,1.969239,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9077-e1aa-7422-ae3b-35801c7a6490
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,1.929586,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9077-e95c-7411-b155-748a74dfca87


In [68]:
evaluate(
    ensemble_retrieval_chain.invoke,
    data=dataset_name,
    metadata={"revision_id": "ensemble_retrieval_chain"},
)

View the evaluation results for experiment: 'best-van-46' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/0b63c101-9dca-4309-bb8f-02b620d02913/compare?selectedSessions=e4810557-cc1b-4dcd-8b3b-b4bba5dad874




0it [00:00, ?it/s]

,inputs.question,outputs.response,outputs.context,error,reference.answer,execution_time,example_id,id
0,Wut are the main recomendashuns in Chapter 14 ...,"content=""Based on Chapter 14 of the guide, the...",[page_content='Tips for building new habits:\n...,None,"Chapter 14 recomends that to support wellness,...",9.519746,3577d19c-58fa-4976-8d8c-28027a8c18ca,019c9077-f876-78c0-aac0-a4e595a80a94
1,what chapter 16 say for headaches?,content='Chapter 16 of the document discusses ...,[page_content='Chapter 16: Managing Headaches ...,None,Chapter 16 say headaches are common and you ca...,4.606640,73930c54-edef-4eb1-a4c7-e80e9839cf75,019c9078-1da6-70f2-bf90-c16aaf6d555e
2,What practical strategies does PART 6 of the g...,"content=""Certainly! PART 6 of the guide offers...",[page_content='Chapter 15: Evening Wind-Down R...,None,PART 6 of the guide addresses common health co...,11.268506,1ef402f6-f435-4056-92db-2ffb5ef1daa3,019c9078-2fa6-7942-bad7-3e253c175459
3,"Whaat aree the benfits of magnesum for sleep, ...","content=""Magnesium can be beneficial for sleep...",[page_content='Types of insomnia:\n- Acute ins...,None,Magnesium supplements are listed as a natural ...,8.391791,8489d26a-c0dd-4523-8b0c-a35649530df0,019c9078-5bab-7973-b22f-779c7cfed133
4,How can chamomile be used to support better sl...,content='Chamomile can be used to support bett...,[page_content='Types of insomnia:\n- Acute ins...,None,Chamomile can be used as an herbal tea as a na...,3.555412,a477005c-3955-4b00-86f4-d49345c480c3,019c9078-7c75-7661-9a95-529b2d193c16
5,How can meditation be incorporated into a busy...,"content=""To incorporate meditation into a busy...",[page_content='PART 4: STRESS MANAGEMENT AND M...,None,Meditation can be incorporated into a busy pro...,6.372398,ba56d476-e29d-461a-993b-defe23267a88,019c9078-8a59-7cd0-8efb-36b28b47cd1c
6,How do you do partial crunches for help with l...,"content=""Partial crunches can help strengthen ...",[page_content='Recommended exercises for lower...,None,Partial crunches is when you lie on your back ...,18.672229,a7d0c9bb-c227-4b11-ae32-6cb937528854,019c9078-a33e-7151-989a-cf13f4e0ddc6
7,Wht are Parshul Crunches and how do yu do them...,"content=""Paragraph: Parshul Crunches are a var...",[page_content='- Partial Crunches: Lie on your...,None,Partial Crunches are an exercise for lower bac...,8.565249,f6d14326-6ec4-415b-9c9a-e01ad072e0dd,019c9078-ec30-7fe0-a133-f47e06b4556f
8,What is the Personal Wellness Guide and how ca...,content='The Personal Wellness Guide is a comp...,[page_content='The Personal Wellness Guide\nA ...,None,The Personal Wellness Guide is a comprehensive...,5.211355,cc646bc9-97d7-4bf2-9d89-84a6c1d5e811,019c9079-0da6-7783-938e-c3ce282495d9


#### Summary and recommendation

**Which retriever is best for this data?**

**Best overall**: Parent document retriever. It gives us the best balance between precision and recall. Context precision is 1.0 (all retrieved chunks are relevant), and context recall is 0.94 (we retrieve almost everything we need). The small-to-big approach, searching with smaller chunks but returning larger parent documents, works very well for this health and wellness corpus. Queries land on the correct section, and the model gets enough surrounding context to answer reliably.

Because we are working in the health and wellness domain, precision is especially important. We want to avoid irrelevant or misleading information. Higher precision means lower risk that the model answers based on noisy or unrelated content.

**Runner-up**: Multi-query retriever. It achieves very high recall (0.98) by reformulating the question in several ways, so it rarely misses relevant information. However, precision drops to 0.60, meaning more irrelevant chunks are retrieved. This increases cost and latency due to additional LLM calls, and in health-related topics, lower precision can be problematic. Ensemble retriever reaches recall 1.0 but has the lowest precision (0.43). It retrieves a lot of irrelevant context. Even though it does not miss information, the extra noise can negatively affect answer quality and efficiency.

**Naive is a solid baseline**: 0.86 recall and 0.77 precision—good balance with no extra cost. Contextual compression slightly improves entity recall compared to naive, but overall recall drops to 0.78. The reranker helps a bit, but it does not significantly change the overall picture.

**Worst fit**: BM25. It has the lowest scores on all metrics (recall 0.13, precision 0.22). Sparse keyword matching doesn’t match how this corpus is queried, semantic (embedding-based) retrieval is clearly better here.


**Latency and cost (from LangSmith chart below; chart order = notebook order):** #1 naive, #2 bm25, #3 contextual_compression, #4 multi_query, #5 parent_document, #6 ensemble. **Naive** (#1): P50 ~3 s, P99 ~11 s, ~$0.0019. **BM25** (#2): P50 ~2 s, P99 ~4 s, ~$0.001 (cheapest and among the fastest). **Contextual compression** (#3): P50 ~2.5 s, P99 ~4 s, ~$0.001. **Multi-query** (#4): P50 ~3.5 s, P99 ~14.5 s, ~$0.0026. **Parent document** (#5): P50 ~2 s, P99 ~5.5 s, ~$0.0015. **Ensemble** (#6): P50 ~8.5 s, P99 ~17 s, ~$0.0034 (slowest and most expensive).

**Conclusion (performance + latency + cost):** **Parent document** (#5) is the best overall: best retrieval quality (precision 1.0, recall 0.94) with good latency (P50 ~2 s, P99 ~5.5 s) and moderate cost (~$0.0015). **Naive** (#1) is a solid alternative (0.86 recall, 0.77 precision) but has higher P99 (~11 s) and cost (~$0.0019). **BM25** (#2) is cheapest and fastest but worst on retrieval. **Ensemble** (#6) is slowest and most expensive (P99 ~17 s, ~$0.0034) and not worth it given its low precision. For this data, **parent document** (#5) is the recommended choice: top quality with the best latency/cost trade-off among the top performers.


**Latency and cost by retriever (from LangSmith):**

![Latency and Cost](latency_cost_charts.png)